## Anime Recommendation System

In [1]:
import pandas as pd
import numpy as np
from typing import List, Tuple
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.optim import Adam
import awswrangler as wr


%matplotlib inline

### reading AnimeList Dataset

In [ ]:
animelist_df = wr.s3.read_csv(
    path="s3://senpai-suggest-datasets/anime/animelist.csv",
    use_threads=True,
    usecols=["user_id", "anime_id", "rating"],
    nrows=1500,
)

In [ ]:
animelist_df.sample(5)

In [ ]:
# save the sample data reead from aws s3 to a local parquet file for future use
animelist_df.to_parquet("data/animelist_sample.parquet", index=False)

### Data Ingestion

In [2]:
# read data from the S3 bucket but only read the first 1000 rows to avoid memory issues
def _ingest_data() -> pd.DataFrame:
    """
    Ingest data from an S3 CSV and return a pandas DataFrame.
    Returns:
        pd.DataFrame: A DataFrame containing the ingested data.
    Raises:
        RuntimeError: If there is an error reading the data from S3.
    """
    try:
        return wr.s3.read_csv(
            path="s3://your-bucket-name/your-file.csv",
            use_threads=True,  # must be keyword argument
            usecols=["user_id", "anime_id", "rating"],
            nrows=1000,
        )
    except Exception as exc:
        raise RuntimeError(f"Failed to read data from S3: {exc}") from exc


In [ ]:
def get_num_ratings(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Get the number of ratings for each user in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[pd.DataFrame, pd.Series]: A tuple containing the filtered dataframe and a series with user_id as index and number of ratings as values.
    """
    if "user_id" not in df.columns:
        raise ValueError("DataFrame must contain 'user_id' column.")
    n_ratings = df["user_id"].value_counts()
    df = df[df["user_id"].isin(n_ratings[n_ratings >= 400].index)].copy()
    return df, n_ratings


In [ ]:
def get_anime_stats(df: pd.DataFrame) -> Tuple[float, float, float]:
    """
    Get the minimum, maximum, and average rating from the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[float, float, float]: A tuple containing minimum rating, maximum rating, and average rating.
    """
    if "rating" not in df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    min_rating = min(df["rating"])
    max_rating = max(df["rating"])
    avg_rating = np.mean(df["rating"])

    return min_rating, max_rating, avg_rating

In [ ]:
def normalize_ratings(
    rating_df: pd.DataFrame,
    min_rating: float,
    max_rating: float,
) -> pd.DataFrame:
    """
    Normalize the ratings in the dataframe to a range of 0 to 1.

    Parameters:
    rating_df (pd.DataFrame): The input dataframe containing user ratings.
    min_rating (float): The minimum rating value.
    max_rating (float): The maximum rating value.

    Returns:
    pd.DataFrame: The dataframe with normalized ratings.
    """
    if "rating" not in rating_df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    scale = max_rating - min_rating
    if scale == 0:
        raise ValueError("max_rating and min_rating cannot be the same.")

    rating_df["rating"] = ((rating_df["rating"] - min_rating) / scale).astype(np.float64)
    return rating_df

In [ ]:
def check_duplicates(df: pd.DataFrame) -> bool:
    """
    Check for duplicate rows in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe to check for duplicates.

    Returns:
    bool: True if duplicates are found, False otherwise.
    """
    return df.duplicated().any()


def check_nulls(df: pd.DataFrame) -> bool:
    """
    Check for null values in the dataframe.

    Parameters:
    df (pd.DataFrame): The input dataframe to check for null values.

    Returns:
    bool: True if null values are found, False otherwise.
    """
    return df.isnull().values.any()

In [ ]:
def encode_users(df: pd.DataFrame) -> Tuple[pd.DataFrame, dict, dict]:
    """
    Encode user IDs in the dataframe to a continuous range of integers.

    Parameters:
    df (pd.DataFrame): The input dataframe containing user ratings.

    Returns:
    Tuple[pd.DataFrame, dict, dict]: A tuple containing the dataframe with encoded user IDs,
                                    a dictionary mapping original user IDs to encoded IDs,
                                    and a dictionary mapping encoded IDs back to original user IDs.
    """
    if "user_id" not in df.columns:
        raise ValueError("DataFrame must contain 'user_id' column.")

    user_ids = df["user_id"].unique().tolist()
    user2user_encoded = {x: i for i, x in enumerate(user_ids)}
    user2user_decoded = {i: x for i, x in enumerate(user_ids)}
    df["user"] = df["user_id"].map(user2user_encoded)

    return df, user2user_encoded, user2user_decoded


In [ ]:
class Preprocessing:
    """class for creating preprocessing steps for the anime data"""

    @staticmethod
    def get_num_ratings(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
        """
            Get the number of ratings for each user in the dataframe.

        Parameters:
            df (pd.DataFrame): The input dataframe containing user ratings.

        Returns:
        Tuple[pd.DataFrame, pd.Series]: A tuple containing the filtered dataframe and a series with user_id as index and number of ratings as values.
        """
        if "user_id" not in df.columns:
            raise ValueError("DataFrame must contain 'user_id' column.")
        n_ratings = df["user_id"].value_counts()
        df = df[df["user_id"].isin(n_ratings[n_ratings >= 400].index)].copy()
        return df, n_ratings

    @staticmethod
    def get_anime_stats(df: pd.DataFrame) -> Tuple[float, float, float]:
        """
            Get the minimum, maximum, and average rating from the dataframe.

        Parameters:
            df (pd.DataFrame): The input dataframe containing user ratings.

        Returns:
        Tuple[float, float, float]: A tuple containing minimum rating, maximum rating, and average rating.
        """
        if "rating" not in df.columns:
            raise ValueError("DataFrame must contain 'rating' column.")

        min_rating = min(df["rating"])
        max_rating = max(df["rating"])
        avg_rating = np.mean(df["rating"])

        return min_rating, max_rating, avg_rating


In [ ]:
# get a sample of the ratings data
from typing import Optional


def get_sample_ratings(
    df: pd.DataFrame,
    sample_size: Optional[int] = 1000,
) -> pd.DataFrame:
    """
    Get a random sample of the ratings data.

    Parameters:
        df (pd.DataFrame): The input dataframe containing user ratings.
        sample_size (int): The number of samples to return. Default is 1000.

    Returns:
        pd.DataFrame: A dataframe containing a random sample of the ratings data.
    """
    if not isinstance(sample_size, int) or sample_size <= 0:
        raise ValueError("sample_size must be a positive integer.")

    if sample_size is None:
        return df.sample(frac=1, random_state=42).reset_index(drop=True)
    if sample_size > len(df):
        raise ValueError("sample_size cannot be greater than the number of rows in the dataframe.")
    return df.sample(n=sample_size, random_state=42).reset_index(drop=True)

In [ ]:
from sklearn.model_selection import train_test_split


# get the dependent and independent variables for the model
def get_features_and_target(df: pd.DataFrame) -> Tuple[np.ndarray, pd.Series]:
    """
    Get the independent variables (features) and dependent variable (target) from the dataframe.

    Parameters:
        df (pd.DataFrame): The input dataframe containing user ratings.
    Returns:
        Tuple[np.ndarray, pd.Series]: A tuple containing the features array and the target series.
    """
    if "rating" not in df.columns:
        raise ValueError("DataFrame must contain 'rating' column.")

    X = df[["user", "anime_id"]].values
    y = df["rating"]
    return X, y


def split_train_test(
    X: np.ndarray,
    y: pd.Series,
    test_size: float = 0.2,
    random_state: int = 42,
) -> Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]:
    """
    Split the features and target into training and testing sets.

    Parameters:
        X (np.ndarray): The features array.
        y (pd.Series): The target series.
        test_size (float): The proportion of the dataset to include in the test split. Default is 0.2.
        random_state (int): The seed used by the random number generator. Default is 42.

    Returns:
        Tuple[np.ndarray, np.ndarray, pd.Series, pd.Series]: A tuple containing the training features, testing features, training target, and testing target.
    """
    if not isinstance(test_size, float) or not (0 < test_size < 1):
        raise ValueError("test_size must be a float between 0 and 1.")

    if not isinstance(random_state, int):
        raise ValueError("random_state must be an integer.")

    return train_test_split(X, y, test_size=test_size, random_state=random_state)